In [1]:
import torch
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, get_peft_model, LoraConfig, TaskType
import os
os.environ["VLLM_USE_V1"] = "0"
import vllm
import sys
import pandas as pd
from datasets import Dataset

import torch.nn.functional as F

model_id = "Qwen/Qwen1.5-1.8B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map={"": torch.cuda.current_device()},
    quantization_config=bnb_config,
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)



data_path = "jigsaw/test.csv" \
                if os.getenv('KAGGLE_IS_COMPETITION_RERUN') \
                else "jigsaw/train.csv"

df = pd.read_csv(data_path)
df.head()



SYS_PROMPT = """
You are given a comment on reddit. Your task is to classify if it violates the given rule. Only respond Yes/No.
"""

def format_example(row):
    prompt_text = f"""
r/{row["subreddit"]}
Rule: {row["rule"]}

1) {row["positive_example_1"]}
Violation: Yes

2) {row["negative_example_1"]}
Violation: No

3) {row["negative_example_2"]}
Violation: No

4) {row["positive_example_2"]}
Violation: Yes

5) {row["body"]}
"""
    messages = [
        {"role": "system", "content": SYS_PROMPT},
        {"role": "user", "content": prompt_text}
    ]
    full_prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    answer = "Yes" if row["rule_violation"] == 1 else "No"
    full_text = full_prompt + "Violation: " + answer

    tokenized = tokenizer(full_text, truncation=True, padding="max_length", max_length=512)
    tokenized["labels"] = tokenized["input_ids"].copy()  # 👈 添加 labels 字段
    return tokenized



hf_dataset = Dataset.from_pandas(df)
tokenized_ds = hf_dataset.map(format_example)


from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-5,
    fp16=True,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds,
    tokenizer=tokenizer,
    data_collator=default_data_collator,
    label_names=["labels"],  # 👈 显式指定
)
trainer.train()



def get_yes_probability(prompt: str, model, tokenizer) -> float:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
    
    # logits: (batch_size, seq_len, vocab_size)
    logits = outputs.logits
    last_token_logits = logits[0, -1]  # 最后一个位置的预测分布
    
    yes_token_id = tokenizer(" Yes", add_special_tokens=False)["input_ids"][0]
    no_token_id = tokenizer(" No", add_special_tokens=False)["input_ids"][0]

    probs = F.softmax(last_token_logits, dim=-1)
    yes_prob = probs[yes_token_id].item()
    no_prob = probs[no_token_id].item()

    # Normalize just in case
    prob = yes_prob / (yes_prob + no_prob + 1e-8)
    return round(prob, 4)


probs = []
for row in df_test.itertuples():
    # 构造 prompt（与训练一致）
    prompt_text = f"""r/{row.subreddit}
Rule: {row.rule}

1) {row.positive_example_1}
Violation: Yes

2) {row.negative_example_1}
Violation: No

3) {row.negative_example_2}
Violation: No

4) {row.positive_example_2}
Violation: Yes

5) {row.body}
Violation:"""

    messages = [
        {"role": "system", "content": SYS_PROMPT},
        {"role": "user", "content": prompt_text}
    ]
    prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    
    prob = get_yes_probability(prompt, model, tokenizer)
    probs.append(prob)


df_test["rule_violation"] = probs
df_test[["row_id", "rule_violation"]].to_csv("submission.csv", index=False)


In [10]:
# 保存 LoRA adapter + tokenizer
model.save_pretrained("./outputs/qwen-lora-reddit")
tokenizer.save_pretrained("./outputs/qwen-lora-reddit")


('./outputs/qwen-lora-reddit/tokenizer_config.json',
 './outputs/qwen-lora-reddit/special_tokens_map.json',
 './outputs/qwen-lora-reddit/chat_template.jinja',
 './outputs/qwen-lora-reddit/vocab.json',
 './outputs/qwen-lora-reddit/merges.txt',
 './outputs/qwen-lora-reddit/added_tokens.json',
 './outputs/qwen-lora-reddit/tokenizer.json')